# Emerging Market v7 — A v E + New Business Loading + Credibility Blend

Key changes from v6:
1. **A v E offset**: `log(expected_claim)` using 2023 LR (first full emerging year)
2. **Year removed** from features
3. **New business loading**: Applies loading factor for 2025 broker portfolio (76% new business vs 14% in training)
4. **Two-part model** alongside Tweedie and GLM
5. **Credibility blend** with developed market v7 predictions
6. Data-driven development (developed count LDFs + emerging severity LDFs)

In [0]:
# Step 1: Load v7 proc files + external LDFs + new business loading
import pandas as pd
import numpy as np
import os, json

%pip install xgboost statsmodels -q

BASE = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026'
DATA_DIR = f'{BASE}/data_emerging/'
V7_DIR = f'{BASE}/files_v7_emerging/'
V7_DEV_DIR = f'{BASE}/files_v7/'
SUB_DIR = f'{BASE}/submissions_v7_emerging/'
os.makedirs(SUB_DIR, exist_ok=True)

# Load LR_2023
with open(os.path.join(V7_DIR, 'lr_2023.json')) as f:
    LR_2023 = {k: float(v) for k, v in json.load(f).items()}
print(f'LR_2023: {LR_2023}')

# === External LDFs for BI/subsidence (same as developed) ===
RAW_LDFS = {
    'bodily_injury': {12: (1.85, 2.40), 24: (1.40, 1.65), 36: (1.20, 1.35), 48: (1.10, 1.18), 60: (1.05, 1.08), 'tail': (1.02, 1.05)},
    'subsidence': {12: (1.50, 1.80), 24: (1.15, 1.25), 36: (1.05, 1.10), 48: (1.01, 1.03), 60: (1.00, 1.00), 'tail': (1.00, 1.00)},
}
LDF_MID = {p: {a: (r[0]+r[1])/2 for a, r in f.items()} for p, f in RAW_LDFS.items()}
SNAPSHOT = pd.Timestamp('2024-12-31')
loss_years = [2021, 2022, 2023, 2024]
dev_ages = {yr: (SNAPSHOT - pd.Timestamp(f'{yr}-07-01')).days / 30.44 for yr in loss_years}

def interp_ldf(peril, age):
    t = LDF_MID[peril]; ages = [12, 24, 36, 48, 60]
    if age <= 12: return t[12]
    elif age >= 60: return t[60] * t['tail']
    for i in range(len(ages)-1):
        if ages[i] <= age <= ages[i+1]:
            frac = (age - ages[i]) / (ages[i+1] - ages[i])
            return t[ages[i]] + frac * (t[ages[i+1]] - t[ages[i]])
    return t[60]

YEAR_LDFS = {yr: {'bodily_injury': interp_ldf('bodily_injury', dev_ages[yr]),
                  'subsidence': interp_ldf('subsidence', dev_ages[yr])} for yr in loss_years}

# Data-driven LDFs (emerging uses developed count LDFs + emerging severity LDFs)
DATA_DRIVEN_LDF = {2021: 1.0, 2022: 1.0, 2023: 1.026, 2024: 1.247}

# Load claims for BI/subsidence scaling
claims = pd.read_parquet(f'{BASE}/data_emerging/claims.parquet')
claims['loss_year'] = pd.to_datetime(claims['loss_date']).dt.year
claims['incurred_net'] = claims['incurred_to_date'] - claims['recoveries']
claims = claims[claims['status'] != 'repudiated'].copy()
claims = claims[claims['loss_year'].isin([2021, 2022, 2023, 2024])].copy()

claims['external_ldf'] = 1.0
for idx, row in claims.iterrows():
    p = str(row['peril']).lower()
    yr = row['loss_year']
    if p == 'bodily_injury':
        claims.at[idx, 'external_ldf'] = YEAR_LDFS.get(yr, {}).get('bodily_injury', 1.0)
    elif p == 'subsidence':
        claims.at[idx, 'external_ldf'] = YEAR_LDFS.get(yr, {}).get('subsidence', 1.0)

claims['data_driven_ldf'] = claims['loss_year'].map(DATA_DRIVEN_LDF).fillna(1.0)
claims['additional_scaling'] = np.where(
    claims['external_ldf'] > 1.0,
    claims['external_ldf'] / claims['data_driven_ldf'],
    1.0
)

bi_subs_claims = claims[claims['additional_scaling'] > 1.0]
print(f'BI/subsidence claims: {len(bi_subs_claims)}')
if len(bi_subs_claims) > 0:
    policy_scaling = bi_subs_claims.groupby(['line', 'policy_id']).apply(
        lambda g: np.average(g['additional_scaling'], weights=g['incurred_net']) if g['incurred_net'].sum() > 0 else 1.0
    ).reset_index(name='scaling_factor')
    print(f'Policies with BI/subsidence scaling: {len(policy_scaling)}')
else:
    policy_scaling = pd.DataFrame(columns=['line', 'policy_id', 'scaling_factor'])

# Update proc files with external LDF scaling
train_v7 = {}
for line in ['motor', 'buildings', 'contents']:
    train_v7[line] = pd.read_parquet(os.path.join(V7_DIR, f'proc_{line}_v7.parquet'))
    old_mean = train_v7[line]['ultimate_claims'].mean()
    ls = policy_scaling[policy_scaling['line'] == line][['policy_id', 'scaling_factor']]
    if len(ls) > 0:
        train_v7[line] = train_v7[line].merge(ls, on='policy_id', how='left')
        train_v7[line]['scaling_factor'] = train_v7[line]['scaling_factor'].fillna(1.0)
        train_v7[line]['ultimate_claims'] = train_v7[line]['ultimate_claims'] * train_v7[line]['scaling_factor']
        train_v7[line].drop('scaling_factor', axis=1, inplace=True)
    new_mean = train_v7[line]['ultimate_claims'].mean()
    print(f'{line}: old mean={old_mean:.4f}, new mean={new_mean:.4f}')

# === NEW BUSINESS LOADING ===
# Compute the claim frequency and severity ratio for new vs renewal business
print('\n=== New Business Loading ===')
NB_LOADING = {}
for line in ['motor', 'buildings', 'contents']:
    df = train_v7[line]
    nb = df[df['new_business'] == True]
    rn = df[df['new_business'] == False]
    # Loss ratio ratio: LR_new / LR_renewal
    lr_nb = nb['ultimate_claims'].sum() / (nb['exposure'] * nb['written_premium']).sum()
    lr_rn = rn['ultimate_claims'].sum() / (rn['exposure'] * rn['written_premium']).sum()
    loading = lr_nb / lr_rn if lr_rn > 0 else 1.0
    NB_LOADING[line] = loading
    print(f'  {line}: LR_new={lr_nb:.4f}, LR_renewal={lr_rn:.4f}, loading={loading:.4f}')
print(f'NB_LOADING: {NB_LOADING}')
print('\nStep 1 complete.')

In [0]:
# Step 2: Test prep + A v E XGBoost (offset = log(expected_claim), Year + written_premium removed)
import xgboost as xgb

test = {}
for line in ['motor', 'buildings', 'contents']:
    test[line] = pd.read_parquet(os.path.join(DATA_DIR, f'test_{line}.parquet'))
customers = pd.read_parquet(os.path.join(DATA_DIR, 'customers.parquet'))
addresses = pd.read_parquet(os.path.join(DATA_DIR, 'addresses.parquet'))
weather = pd.read_parquet(os.path.join(DATA_DIR, 'weather.parquet'))
sample_submission = pd.read_parquet(os.path.join(DATA_DIR, 'sample_submission.parquet'))

# Clean test files (same as data prep)
for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    ps = pd.to_datetime(df['period_start'])
    pe = pd.to_datetime(df['period_end'])
    df['exposure'] = ((pe - ps).dt.days + 1) / 365.0
    df['Year'] = ps.dt.year
    for col in ['policy_id', 'customer_id', 'address_id', 'row_id']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    if 'compulsory_excess' in df.columns:
        df.drop('compulsory_excess', axis=1, inplace=True)

# Sum insured fix
for line in ['buildings', 'contents']:
    df = test[line]
    if 'sum_insured' in df.columns:
        threshold = 10_000_000 if line == 'buildings' else 1_000_000
        flagged = df['sum_insured'] >= threshold
        if flagged.sum() > 0:
            df.loc[flagged, 'sum_insured'] = df.loc[flagged, 'sum_insured'] / 1000
            print(f'test_{line}: {flagged.sum()} sum_insured values fixed')

# Construction type harmonization
CONSTRUCTION_MAP_EMG = {'BRK': 'BRICK', 'STN': 'STONE', 'WOD': 'TIMBER', 'MIX': 'MIXED'}
test['buildings']['construction_type'] = test['buildings']['construction_type'].astype(str).str.strip().str.upper().replace(CONSTRUCTION_MAP_EMG)

# Add age_band if missing
for line in ['motor']:
    if 'driver_age_band' not in test[line].columns and 'driver_age' in test[line].columns:
        test[line]['driver_age_band'] = test[line]['driver_age'].apply(lambda a: '17-24' if a<=24 else '25-34' if a<=34 else '35-49' if a<=49 else '50-64' if a<=64 else '65+')
for line in ['buildings', 'contents']:
    if 'policyholder_age_band' not in test[line].columns and 'policyholder_age' in test[line].columns:
        test[line]['policyholder_age_band'] = test[line]['policyholder_age'].apply(lambda a: '17-24' if a<=24 else '25-34' if a<=34 else '35-49' if a<=49 else '50-64' if a<=64 else '65+')

# Join enrichment
customers_clean = customers.drop(columns=['surname'], errors='ignore')
weather_agg = weather.groupby(['address_id', 'year'], as_index=False).mean(numeric_only=True)
weather_agg.rename(columns={'year': 'Year'}, inplace=True)

for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    df = df.merge(customers_clean, on='customer_id', how='left', suffixes=('', '_cust'))
    df = df.merge(addresses, on='address_id', how='left', suffixes=('', '_addr'))
    df = df.merge(weather_agg, on=['address_id', 'Year'], how='left', suffixes=('', '_weather'))
    df['expected_claim'] = df['exposure'] * df['written_premium'] * LR_2023[line]
    test[line] = df
    print(f'test_{line}: {df.shape}, expected_claim mean={df["expected_claim"].mean():.2f}')

# Feature selection: auto-select (same as v6) but exclude Year + written_premium
EXCLUDE_COLS = {'ultimate_claims', 'row_id', 'policy_id', 'customer_id', 'address_id',
                'period_start', 'period_end', 'claim_count', 'reported_incurred',
                'surname', 'date_of_birth',
                'customer_service_contacts', 'marketing_consent', 'document_preference',
                'campaign_code', 'email_domain',
                'Year', 'written_premium', 'expected_claim', 'ae_ratio',
                'open_claim_count'}

def tweedie_deviance(y, y_pred, p=1.5):
    y = np.array(y, dtype=float)
    y_pred = np.maximum(y_pred, 1e-20)
    return 2 * (np.power(np.maximum(y, 0), 2-p) / ((1-p)*(2-p))
                - y * np.power(y_pred, 1-p) / (1-p)
                + np.power(y_pred, 2-p) / (2-p))

def tweedie_d2(y, y_pred, p=1.5):
    dev = tweedie_deviance(y, y_pred, p).sum()
    dev_null = tweedie_deviance(y, np.mean(y), p).sum()
    return 1 - dev / dev_null

XGB_PARAMS = {
    'objective': 'reg:tweedie', 'tweedie_variance_power': 1.5,
    'max_depth': 3, 'eta': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.5,
    'min_child_weight': 20, 'reg_alpha': 1.0, 'reg_lambda': 3.0,
    'seed': 42, 'n_jobs': -1, 'verbosity': 0,
}
NEW_TREND = {'motor': 1.04, 'buildings': 1.096, 'contents': 1.049}

xgb_val_d2 = {}
xgb_val_preds = {}
xgb_test_preds = {}
xgb_importance = {}

from sklearn.model_selection import KFold

for line in ['motor', 'buildings', 'contents']:
    print(f'\n{"="*60}\n=== {line.upper()} A v E XGBoost v7 ===\n{"="*60}')
    train_df = train_v7[line]
    test_df = test[line]
    candidate_feats = [c for c in train_df.columns if c not in EXCLUDE_COLS and c in test_df.columns]
    feats = [c for c in candidate_feats if train_df[c].isnull().mean() < 0.5]
    print(f'  Features: {len(feats)}')
    
    X_all = train_df[feats].copy()
    X_test = test_df[feats].copy()
    for col in feats:
        if X_all[col].dtype == 'object' or X_all[col].dtype == 'bool':
            cats = pd.Categorical(X_all[col])
            X_all[col] = cats.codes.astype('float64')
            X_test[col] = pd.Categorical(X_test[col], categories=cats.categories).codes.astype('float64')
    
    y_all = train_df['ultimate_claims'].values
    exp_all = train_df['exposure'].values
    offset_all = np.log(train_df['expected_claim'].values + 1e-10)
    offset_test = np.log(test_df['expected_claim'].values + 1e-10)
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    val_preds_all = np.zeros(len(y_all))
    test_preds_folds = []
    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_all)):
        dtrain = xgb.DMatrix(X_all.iloc[tr_idx], label=y_all[tr_idx],
                             base_margin=offset_all[tr_idx], weight=exp_all[tr_idx])
        dval = xgb.DMatrix(X_all.iloc[va_idx], label=y_all[va_idx],
                           base_margin=offset_all[va_idx], weight=exp_all[va_idx])
        dtest = xgb.DMatrix(X_test, base_margin=offset_test)
        model = xgb.train(XGB_PARAMS, dtrain, num_boost_round=200,
                         evals=[(dval, 'val')], early_stopping_rounds=30, verbose_eval=False)
        val_preds_all[va_idx] = np.maximum(model.predict(dval), 0)
        test_preds_folds.append(np.maximum(model.predict(dtest), 0.01))
        if fold == 0:
            scores = model.get_score(importance_type='gain')
            imp = pd.DataFrame({'feature': feats, 'importance': [scores.get(f, 0) for f in feats]}).sort_values('importance', ascending=False)
            xgb_importance[line] = imp
    d2 = tweedie_d2(y_all, val_preds_all)
    xgb_val_d2[line] = d2
    xgb_val_preds[line] = val_preds_all
    xgb_test_preds[line] = np.mean(test_preds_folds, axis=0)
    print(f'  CV D2: {d2:.4f}')
    print(f'  Top 5: {xgb_importance[line].head(5)["feature"].tolist()}')
    print(f'  Test pred mean: {xgb_test_preds[line].mean():.4f}')
print(f'\nA v E XGBoost v7 avg D2: {np.mean(list(xgb_val_d2.values())):.4f}')

In [0]:
# Step 3: Two-part model + GLM (combined cell)
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

# --- Two-part model ---
tp_val_preds = {}
tp_test_preds = {}
tp_val_d2 = {}

FREQ_PARAMS = {'objective': 'binary:logistic', 'max_depth': 3, 'eta': 0.1,
               'subsample': 0.8, 'colsample_bytree': 0.5, 'min_child_weight': 20,
               'reg_alpha': 1.0, 'reg_lambda': 3.0, 'seed': 42, 'n_jobs': -1,
               'verbosity': 0, 'eval_metric': 'logloss'}
SEV_PARAMS = {'objective': 'reg:gamma', 'max_depth': 3, 'eta': 0.05,
              'subsample': 0.8, 'colsample_bytree': 0.5, 'min_child_weight': 10,
              'reg_alpha': 0.5, 'reg_lambda': 2.0, 'seed': 42, 'n_jobs': -1,
              'verbosity': 0, 'eval_metric': 'gamma-deviance'}

for line in ['motor', 'buildings', 'contents']:
    print(f'\n{"="*60}\n=== {line.upper()} Two-Part v7 ===\n{"="*60}')
    train_df = train_v7[line]
    test_df = test[line]
    feats = [c for c in train_df.columns if c not in EXCLUDE_COLS and c in test_df.columns and train_df[c].isnull().mean() < 0.5]
    
    X_all = train_df[feats].copy()
    X_test = test_df[feats].copy()
    for col in feats:
        if X_all[col].dtype == 'object' or X_all[col].dtype == 'bool':
            cats = pd.Categorical(X_all[col])
            X_all[col] = cats.codes.astype('float64')
            X_test[col] = pd.Categorical(X_test[col], categories=cats.categories).codes.astype('float64')
    
    y_all = train_df['ultimate_claims'].values
    y_freq = (y_all > 0).astype(int)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    val_preds_tp = np.zeros(len(y_all))
    test_preds_tp = np.zeros(len(test_df))
    
    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_all)):
        # Frequency
        dtrain_f = xgb.DMatrix(X_all.iloc[tr_idx], label=y_freq[tr_idx], weight=train_df.iloc[tr_idx]['exposure'].values)
        dval_f = xgb.DMatrix(X_all.iloc[va_idx], label=y_freq[va_idx], weight=train_df.iloc[va_idx]['exposure'].values)
        dtest_f = xgb.DMatrix(X_test)
        freq_model = xgb.train(FREQ_PARAMS, dtrain_f, num_boost_round=150, evals=[(dval_f, 'val')],
                               early_stopping_rounds=30, verbose_eval=False)
        freq_val = np.clip(freq_model.predict(dval_f), 0.001, 0.999)
        freq_test = np.clip(freq_model.predict(dtest_f), 0.001, 0.999)
        
        # Severity (on positive claims in training fold)
        pos_tr = y_all[tr_idx] > 0
        if pos_tr.sum() > 30:
            pos_idx = np.where(pos_tr)[0]
            y_sev = np.maximum(y_all[tr_idx][pos_idx], 1.0)
            dtrain_s = xgb.DMatrix(X_all.iloc[tr_idx].iloc[pos_idx], label=y_sev,
                                   weight=train_df.iloc[tr_idx].iloc[pos_idx]['exposure'].values)
            dval_s = xgb.DMatrix(X_all.iloc[va_idx])
            dtest_s = xgb.DMatrix(X_test)
            sev_model = xgb.train(SEV_PARAMS, dtrain_s, num_boost_round=150, evals=[],
                                 early_stopping_rounds=None, verbose_eval=False)
            sev_val = np.maximum(sev_model.predict(dval_s), 1.0)
            sev_test = np.maximum(sev_model.predict(dtest_s), 1.0)
        else:
            mean_sev = max(y_all[tr_idx][pos_tr].mean(), 1.0) if pos_tr.sum() > 0 else 1.0
            sev_val = np.ones(len(va_idx)) * mean_sev
            sev_test = np.ones(len(test_df)) * mean_sev
        
        val_preds_tp[va_idx] = freq_val * sev_val
        test_preds_tp += freq_test * sev_test / 5
    
    d2 = tweedie_d2(y_all, val_preds_tp)
    tp_val_d2[line] = d2
    tp_val_preds[line] = val_preds_tp
    tp_test_preds[line] = np.maximum(test_preds_tp, 0.01)
    print(f'  TP CV D2: {d2:.4f}')
    print(f'  TP Test pred mean: {tp_test_preds[line].mean():.4f}')
print(f'\nTwo-Part v7 avg D2: {np.mean(list(tp_val_d2.values())):.4f}')

# --- GLM with A v E offset ---
SMOOTHING = 30
MAX_FEATS = 20

def target_encode(train_series, train_target, apply_series, smoothing=30):
    prior = train_target.mean()
    grp = pd.DataFrame({'cat': train_series, 'target': train_target})
    stats = grp.groupby('cat')['target'].agg(['mean', 'count'])
    smooth = (stats['mean'] * stats['count'] + prior * smoothing) / (stats['count'] + smoothing)
    return train_series.map(smooth).fillna(prior), apply_series.map(smooth).fillna(prior)

glm_val_d2 = {}
glm_val_preds = {}
glm_test_preds = {}

for line in ['motor', 'buildings', 'contents']:
    print(f'\n{"="*60}\n=== {line.upper()} GLM v7 (A v E) ===\n{"="*60}')
    train_df = train_v7[line]
    test_df = test[line]
    top_feats = xgb_importance[line]['feature'].head(MAX_FEATS).tolist()
    top_feats = [f for f in top_feats if f in train_df.columns and f in test_df.columns and f not in EXCLUDE_COLS]
    print(f'  Features: {len(top_feats)}')
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    val_preds_glm = np.zeros(len(train_df))
    test_preds_glm = np.zeros(len(test_df))
    y_all = train_df['ultimate_claims'].values
    for tr_idx, va_idx in kf.split(train_df):
        y_tr = y_all[tr_idx]
        y_val = y_all[va_idx]
        offset_tr = np.log(train_df.iloc[tr_idx]['expected_claim'].values + 1e-10)
        offset_val = np.log(train_df.iloc[va_idx]['expected_claim'].values + 1e-10)
        offset_test = np.log(test_df['expected_claim'].values + 1e-10)
        X_tr = pd.DataFrame(index=range(len(tr_idx)))
        X_val = pd.DataFrame(index=range(len(va_idx)))
        X_test_g = pd.DataFrame(index=range(len(test_df)))
        for col in top_feats:
            tr_v = train_df.iloc[tr_idx][col]
            va_v = train_df.iloc[va_idx][col]
            te_v = test_df[col]
            if tr_v.dtype == 'object' or tr_v.dtype == 'bool':
                tr_e, va_e = target_encode(tr_v, y_tr, va_v, SMOOTHING)
                _, te_e = target_encode(tr_v, y_tr, te_v, SMOOTHING)
                X_tr[col] = tr_e.values
                X_val[col] = va_e.values
                X_test_g[col] = te_e.values
            else:
                med = tr_v.median()
                X_tr[col] = tr_v.fillna(med).values
                X_val[col] = va_v.fillna(med).values
                X_test_g[col] = te_v.fillna(med).values
        scaler = StandardScaler()
        X_tr_s = sm.add_constant(pd.DataFrame(scaler.fit_transform(X_tr), columns=X_tr.columns), has_constant='add')
        X_val_s = sm.add_constant(pd.DataFrame(scaler.transform(X_val), columns=X_val.columns), has_constant='add')
        X_test_s = sm.add_constant(pd.DataFrame(scaler.transform(X_test_g), columns=X_test_g.columns), has_constant='add')
        try:
            model = sm.GLM(y_tr, X_tr_s, family=sm.families.Tweedie(link=sm.families.links.Log(), var_power=1.5), offset=offset_tr)
            result = model.fit(max_iter=100)
            val_preds_glm[va_idx] = np.maximum(result.predict(X_val_s, offset=offset_val), 0)
            test_preds_glm += np.maximum(result.predict(X_test_s, offset=offset_test), 0.01) / 5
        except:
            val_preds_glm[va_idx] = y_tr.mean()
            test_preds_glm += y_tr.mean() / 5
    d2 = tweedie_d2(y_all, val_preds_glm)
    glm_val_d2[line] = d2
    glm_val_preds[line] = val_preds_glm
    glm_test_preds[line] = test_preds_glm
    print(f'  GLM CV D2: {d2:.4f}')
print(f'\nGLM v7 avg D2: {np.mean(list(glm_val_d2.values())):.4f}')

In [0]:
# Step 4: Blend + credibility blend (NO NB loading - model already handles NB via feature) + trend + calibration + submission
# NOTE: new_business is already a model feature, so NB loading would double-count.
# The model learns NB effect directly. We just calibrate to the overall training mean.

# 3-way blend (Tweedie + Two-Part + GLM)
print('=== BLEND v7 (Tweedie + Two-Part + GLM) ===')
best_weights = {}
blend_d2 = {}
for line in ['motor', 'buildings', 'contents']:
    y_all = train_v7[line]['ultimate_claims'].values
    best_d2 = -999
    best_w = (0.33, 0.33, 0.34)
    for wx in np.arange(0, 1.01, 0.1):
        for wt in np.arange(0, 1.01 - wx, 0.1):
            wg = 1.0 - wx - wt
            if wg < 0: continue
            blend = wx * xgb_val_preds[line] + wt * tp_val_preds[line] + wg * glm_val_preds[line]
            d2 = tweedie_d2(y_all, blend)
            if d2 > best_d2:
                best_d2 = d2
                best_w = (wx, wt, wg)
    blend_d2[line] = best_d2
    best_weights[line] = best_w
    print(f'{line}: Tweedie={best_w[0]:.2f}, TwoPart={best_w[1]:.2f}, GLM={best_w[2]:.2f} -> D2={best_d2:.4f}')
print(f'Blend v7 avg D2: {np.mean(list(blend_d2.values())):.4f}')

# Generate raw test predictions with blend (NO NB loading)
raw_test_preds = {}
for line in ['motor', 'buildings', 'contents']:
    wx, wt, wg = best_weights[line]
    raw_test_preds[line] = np.clip(wx * xgb_test_preds[line] + wt * tp_test_preds[line] + wg * glm_test_preds[line], 0.01, 50000)

# Load developed priors
dev_priors = {}
for line in ['motor', 'buildings', 'contents']:
    dev_priors[line] = pd.read_parquet(os.path.join(V7_DEV_DIR, f'proc_{line}_v7.parquet'))['ultimate_claims'].mean()

# === GENERATE MULTIPLE SUBMISSIONS with different alpha values ===
for ALPHA in [0.5, 0.7, 1.0]:
    print(f'\n{"="*60}\n=== Alpha={ALPHA} ===\n{"="*60}')
    blend_test_preds = {}
    for line in ['motor', 'buildings', 'contents']:
        n_emerging = train_v7[line]['exposure'].sum()
        k = n_emerging * (1 - ALPHA) / ALPHA if ALPHA > 0 else 1e10
        Z = n_emerging / (n_emerging + k)
        cred_pred = Z * raw_test_preds[line] + (1 - Z) * dev_priors[line]
        blend_test_preds[line] = np.clip(cred_pred, 0.01, 50000)
        print(f'  {line}: Z={Z:.4f}, raw_mean={raw_test_preds[line].mean():.4f}, dev_prior={dev_priors[line]:.4f}, cred_mean={blend_test_preds[line].mean():.4f}')
    
    # Trend + calibration (simple: calibrate to 2024 training mean * trend)
    submission = sample_submission.copy()
    for line in ['motor', 'buildings', 'contents']:
        test_df = test[line]
        pred = np.clip(blend_test_preds[line], 0.01, 50000) * NEW_TREND[line]
        # Simple calibration to 2024 training mean * trend (NO NB adjustment)
        train_df = train_v7[line]
        cal_target = train_df.loc[train_df['Year'] == 2024, 'ultimate_claims'].mean() * NEW_TREND[line]
        if cal_target > 0:
            cal_factor = cal_target / max(pred.mean(), 0.01)
            pred = np.clip(pred * cal_factor, 0.01, 50000)
        print(f'  {line}: pred mean={pred.mean():.4f} (target={cal_target:.4f})')
        pred_map = dict(zip(test_df['row_id'], pred))
        mask = submission['row_id'].isin(pred_map)
        submission.loc[mask, 'prediction'] = submission.loc[mask, 'row_id'].map(pred_map)
    
    submission['prediction'] = submission['prediction'].clip(lower=0.01, upper=50000)
    submission['prediction'] = submission['prediction'].replace([np.inf, -np.inf], 50000).fillna(0.01)
    sub_path = os.path.join(SUB_DIR, f'blend_v7_emerging_noNB_alpha{ALPHA}.csv')
    submission.to_csv(sub_path, index=False)
    print(f'  Saved: {sub_path} ({len(submission)} rows, mean={submission["prediction"].mean():.4f})')

# Also save the original (with NB loading) for comparison reference
print(f'\nOriginal v7 (with NB loading): mean=286.80, Kaggle D2=0.12230')
print(f'v6 (no NB loading, alpha=0.5): mean=327.57, Kaggle D2=0.14166')
print(f'\n=== v7 Emerging Market (no NB loading) Complete ===')